# Moduł 10: Zaawansowane Computer Vision

Tematy wykraczające poza podstawowe CNN — **wymagane przez syllabus IOAI 2026** (kategoria Practice).

## Spis treści
1. Generative Adversarial Networks (GANs)
2. Self-Supervised Learning for Vision
3. Vision-Text Encoders (CLIP)
4. Modele dyfuzyjne (Diffusion Models)
5. Zaawansowana detekcja obiektów (DETR, SSD)
6. Przetwarzanie wideo (Video Understanding)
7. **Zadania**

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

---

## 1. Generative Adversarial Networks (GANs)

### Idea

GAN składa się z dwóch sieci trenowanych jednocześnie w grze **minimax**:

- **Generator** $G(z)$: generuje fałszywe dane z szumu $z \sim \mathcal{N}(0, I)$
- **Dyskryminator** $D(x)$: ocenia prawdopodobieństwo, że $x$ pochodzi z danych rzeczywistych

### Funkcja straty GAN

$$\min_G \max_D \; V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

W praktyce trening przebiega naprzemiennie:

**Krok dyskryminatora** (maksymalizuj $V$):
$$\mathcal{L}_D = -\mathbb{E}_{x}[\log D(x)] - \mathbb{E}_{z}[\log(1 - D(G(z)))]$$

**Krok generatora** (minimalizuj, w praktyce maksymalizuj $\log D(G(z))$):
$$\mathcal{L}_G = -\mathbb{E}_{z}[\log D(G(z))]$$

> **Non-saturating loss**: zamiast minimalizować $\log(1-D(G(z)))$, generator maksymalizuje $\log D(G(z))$ — lepsze gradienty na początku treningu.

### Optymalne rozwiązanie

Gdy $G^*$ jest optymalny: $p_G = p_{\text{data}}$, a $D^*(x) = \frac{1}{2}$ dla każdego $x$.

### Warianty GAN

| Wariant | Kluczowa zmiana | Zastosowanie |
|---------|----------------|-------------|
| **DCGAN** | Konwolucje zamiast FC, BatchNorm | Generacja obrazów |
| **WGAN** | Wasserstein distance zamiast JS divergence | Stabilniejszy trening |
| **WGAN-GP** | Gradient penalty zamiast weight clipping | Jeszcze stabilniejszy |
| **Conditional GAN (cGAN)** | Warunkowanie na etykiecie $y$: $G(z, y)$ | Generacja z kontrolą klasy |
| **StyleGAN** | Mapping network + style injection | Fotorealistyczne twarze |
| **Pix2Pix** | Paired image-to-image translation | Tłumaczenie obrazów |
| **CycleGAN** | Unpaired translation, cycle consistency | Styl artystyczny |

### Problemy treningu GAN

- **Mode collapse**: generator produkuje tylko kilka wariantów
- **Training instability**: oscylacje strat, brak konwergencji
- **Vanishing gradients**: gdy D jest zbyt dobry, gradienty G → 0

### Metryki oceny

- **FID (Fréchet Inception Distance)**: porównuje rozkłady cech z Inception. Niższy = lepszy.

$$\text{FID} = \|\mu_r - \mu_g\|^2 + \text{Tr}(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2})$$

- **IS (Inception Score)**: mierzy jakość i różnorodność. Wyższy = lepszy.

In [ ]:
# === Prosty GAN na MNIST ===

# Parametry
latent_dim = 64
img_size = 28 * 28 # MNIST spłaszczony
lr = 0.0002
batch_size = 128

# Generator
class Generator(nn.Module):
 def __init__(self, latent_dim, img_size):
 super().__init__()
 self.net = nn.Sequential(
 nn.Linear(latent_dim, 256),
 nn.LeakyReLU(0.2),
 nn.Linear(256, 512),
 nn.LeakyReLU(0.2),
 nn.Linear(512, img_size),
 nn.Tanh() # output w [-1, 1]
 )

 def forward(self, z):
 return self.net(z)

# Dyskryminator
class Discriminator(nn.Module):
 def __init__(self, img_size):
 super().__init__()
 self.net = nn.Sequential(
 nn.Linear(img_size, 512),
 nn.LeakyReLU(0.2),
 nn.Dropout(0.3),
 nn.Linear(512, 256),
 nn.LeakyReLU(0.2),
 nn.Dropout(0.3),
 nn.Linear(256, 1),
 nn.Sigmoid()
 )

 def forward(self, x):
 return self.net(x)

# Inicjalizacja
G = Generator(latent_dim, img_size)
D = Discriminator(img_size)
opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
criterion = nn.BCELoss()

print(f"Generator params: {sum(p.numel() for p in G.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in D.parameters()):,}")

In [ ]:
# Pętla treningowa GAN (skrócona wersja — demonstracja)
transform = transforms.Compose([
 transforms.ToTensor(),
 transforms.Normalize([0.5], [0.5]) # → [-1, 1]
])

dataset = torchvision.datasets.MNIST(root='./data', train=True,
 download=True, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

num_epochs = 5 # zwiększ do 50+ dla dobrych wyników

for epoch in range(num_epochs):
 for i, (real_imgs, _) in enumerate(dataloader):
 real_imgs = real_imgs.view(-1, img_size)
 bs = real_imgs.size(0)

 # --- Trening dyskryminatora ---
 real_labels = torch.ones(bs, 1)
 fake_labels = torch.zeros(bs, 1)

 # Real
 out_real = D(real_imgs)
 loss_real = criterion(out_real, real_labels)

 # Fake
 z = torch.randn(bs, latent_dim)
 fake_imgs = G(z).detach()
 out_fake = D(fake_imgs)
 loss_fake = criterion(out_fake, fake_labels)

 loss_D = loss_real + loss_fake
 opt_D.zero_grad()
 loss_D.backward()
 opt_D.step()

 # --- Trening generatora ---
 z = torch.randn(bs, latent_dim)
 fake_imgs = G(z)
 out = D(fake_imgs)
 loss_G = criterion(out, real_labels) # non-saturating loss

 opt_G.zero_grad()
 loss_G.backward()
 opt_G.step()

 print(f"Epoch [{epoch+1}/{num_epochs}] | D loss: {loss_D.item():.4f} | G loss: {loss_G.item():.4f}")

# Generowanie próbek
with torch.no_grad():
 z = torch.randn(16, latent_dim)
 samples = G(z).view(-1, 1, 28, 28)
 fig, axes = plt.subplots(2, 8, figsize=(12, 3))
 for i, ax in enumerate(axes.flat):
 ax.imshow(samples[i, 0].numpy(), cmap='gray')
 ax.axis('off')
 plt.suptitle('Wygenerowane cyfry (GAN)')
 plt.tight_layout()
 plt.show()

---

## 2. Self-Supervised Learning for Vision

**Self-supervised learning (SSL)** to uczenie bez etykiet — model uczy się **reprezentacji** z samych danych, tworząc zadania pretekstowe (pretext tasks).

### Dlaczego SSL?
- Etykietowanie danych jest drogie i czasochłonne
- SSL pozwala wyuczyć dobre embeddingi na dużych nieoznaczonych zbiorach
- Potem fine-tuning na małym zbiorze z etykietami (transfer learning)

### Podejścia SSL w computer vision

#### A) Contrastive Learning

Idea: **podobne** obrazy (augmentacje tego samego) powinny mieć **bliskie** embeddingi, **różne** — dalekie.

**SimCLR** (Simple Framework for Contrastive Learning):
1. Weź obraz $x$, stwórz 2 augmentacje: $\tilde{x}_i, \tilde{x}_j$
2. Przepuść przez encoder $f(\cdot)$ → reprezentacje $h_i, h_j$
3. Przepuść przez projection head $g(\cdot)$ → $z_i, z_j$
4. Minimalizuj **NT-Xent loss** (Normalized Temperature-scaled Cross Entropy):

$$\ell_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k) / \tau)}$$

gdzie $\text{sim}(u, v) = \frac{u^T v}{\|u\| \|v\|}$ (cosine similarity), $\tau$ — temperatura.

**MoCo** (Momentum Contrast):
- Używa **momentum encoder** (wolno aktualizowany) i **kolejki** negatywnych przykładów
- Pozwala na duży batch negatywnych bez dużej pamięci GPU
- $\theta_k \leftarrow m \cdot \theta_k + (1 - m) \cdot \theta_q$ (momentum update, $m = 0.999$)

#### B) Non-Contrastive Methods

**BYOL** (Bootstrap Your Own Latent):
- Dwie sieci: online + target (momentum)
- Nie potrzebuje negatywnych przykładów!
- Online network ma dodatkowy predictor head

**DINO** (Self-Distillation with No Labels):
- Teacher-student framework z Vision Transformer
- Centering + sharpening zamiast negatywnych przykładów
- Emergent: attention maps odpowiadają segmentacji obiektów

#### C) Masked Image Modeling

**MAE** (Masked Autoencoders):
- Maskuj 75% patchy obrazu
- Encoder (ViT) przetwarza tylko **widoczne** patche
- Decoder rekonstruuje zamaskowane patche
- Analogia do BERT (MLM) dla obrazów

### Porównanie metod SSL

| Metoda | Typ | Negatywne przykłady? | Architektura |
|--------|-----|---------------------|-------------|
| SimCLR | Contrastive | Tak (duży batch) | ResNet + MLP head |
| MoCo v2 | Contrastive | Tak (kolejka) | ResNet + momentum |
| BYOL | Non-contrastive | Nie | ResNet + predictor |
| DINO | Self-distillation | Nie | ViT + teacher |
| MAE | Masked modeling | Nie | ViT encoder-decoder |

In [ ]:
# === Demonstracja: Augmentacje stosowane w SSL (SimCLR-style) ===

ssl_transforms = transforms.Compose([
 transforms.RandomResizedCrop(32, scale=(0.2, 1.0)),
 transforms.RandomHorizontalFlip(p=0.5),
 transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
 transforms.RandomGrayscale(p=0.2),
 transforms.GaussianBlur(kernel_size=3),
 transforms.ToTensor(),
 transforms.Normalize([0.4914, 0.4822, 0.4465], [0.247, 0.243, 0.262])
])

# Wizualizacja: ten sam obraz, różne augmentacje
from torchvision.datasets import CIFAR10
dataset_raw = CIFAR10(root='./data', train=True, download=True)
img, label = dataset_raw[0]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes[0, 0].imshow(img)
axes[0, 0].set_title('Oryginał')
axes[0, 0].axis('off')

for i in range(1, 10):
 ax = axes[i // 5, i % 5]
 aug_img = ssl_transforms(img)
 # Denormalize for display
 aug_img_display = aug_img * torch.tensor([0.247, 0.243, 0.262]).view(3,1,1) + torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1)
 ax.imshow(aug_img_display.permute(1, 2, 0).clamp(0, 1).numpy())
 ax.set_title(f'Aug {i}')
 ax.axis('off')

plt.suptitle('Augmentacje SimCLR-style')
plt.tight_layout()
plt.show()

In [ ]:
# === NT-Xent Loss (SimCLR) — implementacja ===

def nt_xent_loss(z_i, z_j, temperature=0.5):
 """
 Normalized Temperature-scaled Cross Entropy Loss.
 z_i, z_j: [batch_size, projection_dim] — embeddingi dwóch augmentacji
 """
 batch_size = z_i.size(0)

 # Łączymy obie augmentacje
 z = torch.cat([z_i, z_j], dim=0) # [2*batch_size, dim]

 # Macierz cosine similarity
 z_norm = F.normalize(z, dim=1)
 sim_matrix = torch.mm(z_norm, z_norm.t()) / temperature # [2N, 2N]

 # Maska: usuń diagonalę (sim z sobą)
 mask = ~torch.eye(2 * batch_size, dtype=torch.bool)
 sim_matrix = sim_matrix.masked_select(mask).view(2 * batch_size, -1)

 # Pozytywne pary: (i, i+N) i (i+N, i)
 positives = torch.cat([
 torch.sum(z_norm[:batch_size] * z_norm[batch_size:], dim=1),
 torch.sum(z_norm[batch_size:] * z_norm[:batch_size], dim=1)
 ]) / temperature # [2N]

 # Loss: -log(exp(pos) / sum(exp(all)))
 logits = torch.cat([positives.unsqueeze(1), sim_matrix], dim=1) # [2N, 2N]
 labels = torch.zeros(2 * batch_size, dtype=torch.long) # pozytywna para to indeks 0

 loss = F.cross_entropy(logits, labels)
 return loss

# Test
z_i = torch.randn(32, 128)
z_j = z_i + 0.1 * torch.randn(32, 128) # podobne embeddingi (symulacja pozytywnej pary)
loss = nt_xent_loss(z_i, z_j)
print(f"NT-Xent loss (similar pairs): {loss.item():.4f}")

z_j_random = torch.randn(32, 128) # losowe (niepodobne)
loss_random = nt_xent_loss(z_i, z_j_random)
print(f"NT-Xent loss (random pairs): {loss_random.item():.4f}")
print("→ Loss mniejszy gdy pary są podobne (tak powinno być)")

---

## 3. CLIP — Vision-Text Encoders

**CLIP** (Contrastive Language-Image Pre-training, OpenAI 2021) to model łączący **obraz** i **tekst** w jednej wspólnej przestrzeni embeddingów.

### Architektura

$$\text{CLIP} = \text{Image Encoder} + \text{Text Encoder}$$

- **Image Encoder**: ResNet lub ViT → embedding $v_i \in \mathbb{R}^d$
- **Text Encoder**: Transformer → embedding $t_j \in \mathbb{R}^d$

### Trening — Contrastive Loss

Dla batcha $N$ par (obraz, tekst):

$$\mathcal{L} = -\frac{1}{2N} \sum_{i=1}^{N} \left[ \log \frac{\exp(v_i^T t_i / \tau)}{\sum_{j=1}^{N} \exp(v_i^T t_j / \tau)} + \log \frac{\exp(t_i^T v_i / \tau)}{\sum_{j=1}^{N} \exp(t_i^T v_j / \tau)} \right]$$

- Symetryczny cross-entropy: obraz→tekst + tekst→obraz
- Trenowany na 400M par (obraz, tekst) z internetu

### Zero-shot Classification z CLIP

1. Zakoduj obraz: $v = f_{\text{img}}(\text{image})$
2. Zakoduj etykiety jako tekst: $t_k = f_{\text{text}}(\texttt{"a photo of a [class\_k]"}$)
3. Wybierz klasę z największym cosine similarity:

$$\hat{y} = \arg\max_k \frac{v^T t_k}{\|v\| \|t_k\|}$$

### Zastosowania CLIP

| Zastosowanie | Opis |
|-------------|------|
| Zero-shot classification | Klasyfikacja bez treningu, tylko opisy klas |
| Image retrieval | Wyszukiwanie obrazów po opisie tekstowym |
| Generative guidance | CLIP jako loss do sterowania generacją (np. DALL-E, Stable Diffusion) |
| Multimodal embeddings | Wspólna przestrzeń dla wizji i języka |

In [ ]:
# === CLIP — użycie z Hugging Face (jeśli dostępny) ===
# Na olimpiadzie CLIP jest dostępny offline — załadowany z lokalnego modelu

try:
 from transformers import CLIPProcessor, CLIPModel

 model_name = "openai/clip-vit-base-patch32"
 clip_model = CLIPModel.from_pretrained(model_name)
 clip_processor = CLIPProcessor.from_pretrained(model_name)

 # Zero-shot classification
 # Wczytaj przykładowy obraz
 from torchvision.datasets import CIFAR10
 cifar = CIFAR10(root='./data', train=False, download=True)
 img, true_label = cifar[0]
 class_names = cifar.classes # ['airplane', 'automobile', ...]

 # Przygotuj teksty klas
 text_prompts = [f"a photo of a {c}" for c in class_names]

 inputs = clip_processor(text=text_prompts, images=img, return_tensors="pt", padding=True)

 with torch.no_grad():
 outputs = clip_model(**inputs)
 logits = outputs.logits_per_image # [1, num_classes]
 probs = logits.softmax(dim=-1)

 print("Zero-shot CLIP classification:")
 for name, prob in sorted(zip(class_names, probs[0].tolist()), key=lambda x: -x[1]):
 print(f" {name:15s}: {prob:.4f}")
 print(f"\nTrue label: {class_names[true_label]}")
 print(f"CLIP prediction: {class_names[probs[0].argmax().item()]}")

except ImportError:
 print("transformers nie jest zainstalowany.")
 print("pip install transformers")
 print("\nNa olimpiadzie: model CLIP jest zazwyczaj dostępny offline.")

---

## 4. Modele dyfuzyjne (Diffusion Models)

### Idea

Modele dyfuzyjne generują obrazy przez **stopniowe usuwanie szumu** (denoising). Składają się z dwóch procesów:

### Forward Process (dodawanie szumu)

Stopniowo dodajemy szum gaussowski w $T$ krokach:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t} \, x_{t-1}, \beta_t I)$$

gdzie $\beta_t$ — harmonogram szumu (noise schedule), zwykle $\beta_1 < \beta_2 < \ldots < \beta_T$.

Dzięki właściwości Gaussa, możemy przejść bezpośrednio do kroku $t$:

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t} \, x_0, (1-\bar{\alpha}_t) I)$$

gdzie $\alpha_t = 1 - \beta_t$ i $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$.

To znaczy: $x_t = \sqrt{\bar{\alpha}_t} \, x_0 + \sqrt{1-\bar{\alpha}_t} \, \epsilon$, gdzie $\epsilon \sim \mathcal{N}(0, I)$.

### Reverse Process (usuwanie szumu)

Sieć neuronowa $\epsilon_\theta(x_t, t)$ uczy się **predykować szum** dodany w kroku $t$:

$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon} \left[ \|\epsilon - \epsilon_\theta(x_t, t)\|^2 \right]$$

Sampling (generacja) — iteracyjne usuwanie szumu:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z$$

gdzie $z \sim \mathcal{N}(0, I)$.

### DDPM vs DDIM

| Cecha | DDPM | DDIM |
|-------|------|------|
| Kroki samplingowe | $T$ (np. 1000) | Dowolne (np. 50) |
| Stochastyczność | Tak (dodaje szum) | Deterministyczny |
| Prędkość | Wolny | Szybszy |

### Latent Diffusion (Stable Diffusion)

Zamiast pracować w przestrzeni pikseli ($256 \times 256 \times 3$), dyfuzja odbywa się w **latent space** autoenkoder:

1. **Encoder** VAE: obraz → latent $z_0$ (np. $32 \times 32 \times 4$)
2. **Dyfuzja** w latent space: $z_0 \to z_T \to z_0$ (U-Net z attention)
3. **Decoder** VAE: latent → obraz

Warunkowanie na tekście: cross-attention z embeddingami CLIP/T5.

### Classifier-Free Guidance

$$\hat{\epsilon} = \epsilon_\theta(x_t, \varnothing) + s \cdot (\epsilon_\theta(x_t, c) - \epsilon_\theta(x_t, \varnothing))$$

gdzie $s > 1$ — guidance scale, $c$ — warunek (tekst), $\varnothing$ — brak warunku.

In [ ]:
# === Forward Process — wizualizacja dodawania szumu ===

def linear_beta_schedule(timesteps, beta_start=0.0001, beta_end=0.02):
 return torch.linspace(beta_start, beta_end, timesteps)

T = 1000
betas = linear_beta_schedule(T)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

def forward_diffusion(x_0, t, alphas_cumprod):
 """Dodaj szum do obrazu x_0 w kroku t."""
 noise = torch.randn_like(x_0)
 sqrt_alpha_bar = torch.sqrt(alphas_cumprod[t])
 sqrt_one_minus = torch.sqrt(1.0 - alphas_cumprod[t])
 x_t = sqrt_alpha_bar * x_0 + sqrt_one_minus * noise
 return x_t, noise

# Wizualizacja: obraz w różnych krokach szumu
from torchvision.datasets import CIFAR10
cifar = CIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor())
x_0 = cifar[0][0] # [3, 32, 32]
x_0 = x_0 * 2 - 1 # normalize to [-1, 1]

timesteps_to_show = [0, 50, 100, 250, 500, 750, 999]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(16, 3))

for i, t in enumerate(timesteps_to_show):
 x_t, _ = forward_diffusion(x_0, t, alphas_cumprod)
 img_display = (x_t + 1) / 2 # back to [0, 1]
 axes[i].imshow(img_display.permute(1, 2, 0).clamp(0, 1).numpy())
 axes[i].set_title(f't={t}')
 axes[i].axis('off')

plt.suptitle('Forward Diffusion Process')
plt.tight_layout()
plt.show()

# Wykres ᾱ_t
plt.figure(figsize=(8, 4))
plt.plot(alphas_cumprod.numpy())
plt.xlabel('Timestep t')
plt.ylabel('ᾱ_t (cumulative alpha)')
plt.title('Noise Schedule — im dalej, tym więcej szumu')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# === Prosty model denoising (U-Net uproszczony) ===

class SimpleUNet(nn.Module):
 """Uproszczony U-Net do predykcji szumu."""
 def __init__(self, in_channels=3, time_emb_dim=32):
 super().__init__()
 # Time embedding
 self.time_mlp = nn.Sequential(
 nn.Linear(1, time_emb_dim),
 nn.ReLU(),
 nn.Linear(time_emb_dim, time_emb_dim)
 )
 # Encoder
 self.enc1 = nn.Sequential(nn.Conv2d(in_channels, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64))
 self.enc2 = nn.Sequential(nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(), nn.BatchNorm2d(128))
 # Bottleneck
 self.bottleneck = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.ReLU())
 # Decoder
 self.dec1 = nn.Sequential(nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU())
 self.dec2 = nn.Conv2d(128, in_channels, 3, padding=1) # skip connection: 64+64=128

 def forward(self, x, t):
 # t: [batch] → [batch, 1]
 t_emb = self.time_mlp(t.float().unsqueeze(-1) / 1000.0) # normalize
 # Encoder
 e1 = self.enc1(x)
 e2 = self.enc2(e1)
 # Bottleneck + time conditioning
 b = self.bottleneck(e2) + t_emb[:, :, None, None].expand_as(self.bottleneck(e2))[:, :128]
 # Decoder + skip
 d1 = self.dec1(b)
 d1 = torch.cat([d1, e1], dim=1) # skip connection
 out = self.dec2(d1)
 return out

unet = SimpleUNet()
print(f"Simple U-Net params: {sum(p.numel() for p in unet.parameters()):,}")
print("\n Na olimpiadzie: raczej użyjesz gotowego diffusers pipeline niż pisania U-Net od zera.")

---

## 5. Zaawansowana detekcja obiektów

### SSD (Single Shot MultiBox Detector)

- **Jednoprzebiegowy** (single-shot) — bez region proposal
- Predykcja na **wielu skalach** (multi-scale feature maps)
- Anchor boxes o różnych aspect ratios na każdym level

Architektura:
1. Backbone (np. VGG16) → feature maps
2. Dodatkowe warstwy konwolucyjne zmniejszające resolution
3. Na każdej skali: conv $3 \times 3$ → predykcja klasy + offset bounding box
4. Non-Maximum Suppression (NMS) na końcu

Loss SSD:
$$\mathcal{L} = \frac{1}{N} (\mathcal{L}_{\text{conf}} + \alpha \mathcal{L}_{\text{loc}})$$

- $\mathcal{L}_{\text{conf}}$: cross-entropy (klasyfikacja)
- $\mathcal{L}_{\text{loc}}$: Smooth L1 (regresja bounding box)
- Hard Negative Mining: stosunek negative:positive = 3:1

### DETR (DEtection TRansformer)

**Rewolucja**: detekcja jako problem **set prediction** — bez anchor boxes i NMS!

Architektura:
1. **CNN backbone** (ResNet-50) → feature map
2. **Transformer Encoder** — self-attention na feature map
3. **Transformer Decoder** + **object queries** ($N$ uczonych wektorów, np. $N=100$)
4. Każdy query → predykcja (klasa, bounding box) lub ∅ (no object)

**Hungarian Matching**: bipartite matching między predykcjami a ground truth:

$$\hat{\sigma} = \arg\min_{\sigma \in \mathfrak{S}_N} \sum_{i=1}^{N} \mathcal{L}_{\text{match}}(y_i, \hat{y}_{\sigma(i)})$$

- Matching cost łączy classification loss + box loss (L1 + GIoU)
- **Brak NMS** — każdy query odpowiada za jeden obiekt
- **Brak anchor boxes** — predykcja bezpośrednio w przestrzeni $(c_x, c_y, w, h)$

### Porównanie detektorów

| Metoda | Anchor? | NMS? | Architektura | mAP (COCO) |
|--------|---------|------|-------------|------------|
| Faster R-CNN | Tak | Tak | Two-stage | ~42 |
| YOLO v5/v8 | Tak | Tak | Single-stage | ~50+ |
| SSD | Tak | Tak | Single-stage | ~25-30 |
| DETR | **Nie** | **Nie** | Transformer | ~42 |
| RT-DETR | **Nie** | **Nie** | Transformer | ~54 |

In [ ]:
# === DETR — użycie pretrained modelu ===
try:
 from transformers import DetrImageProcessor, DetrForObjectDetection

 # Załaduj model
 detr_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
 detr_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
 detr_model.eval()

 # Przykład detekcji
 from PIL import Image
 import requests
 # Na olimpiadzie: wczytaj obraz z dysku
 # img = Image.open("path/to/image.jpg")

 # Demo: sztuczczny obraz
 img = Image.fromarray(np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8))

 inputs = detr_processor(images=img, return_tensors="pt")
 with torch.no_grad():
 outputs = detr_model(**inputs)

 # Post-processing: filtruj predykcje z confidence > 0.5
 target_sizes = torch.tensor([img.size[::-1]])
 results = detr_processor.post_process_object_detection(
 outputs, target_sizes=target_sizes, threshold=0.5
 )[0]

 print(f"Wykryto {len(results['scores'])} obiektów (threshold=0.5)")
 for score, label, box in zip(results['scores'], results['labels'], results['boxes']):
 print(f" {detr_model.config.id2label[label.item()]:15s} | conf={score:.3f} | box={box.tolist()}")

 print("\n Wzorzec olimpiadowy: DETR jest dostępny offline z transformers")

except ImportError:
 print("transformers nie jest zainstalowany.")
 print("pip install transformers timm")

## 6. Przetwarzanie wideo (Video Understanding)

Wideo to **sekwencja klatek** (frames) — łączy przetwarzanie obrazów z modelowaniem czasowym.

### 6.1 Reprezentacja wideo

Wideo o $T$ klatkach, rozdzielczości $H \times W$, z $C$ kanałami kolorów:

$$\mathbf{V} \in \mathbb{R}^{T \times C \times H \times W}$$

Typowo: $T \in [16, 64]$ klatki, $H = W = 224$, $C = 3$ (RGB), FPS = 24-30.

### 6.2 Podejścia do rozumienia wideo

| Podejście | Opis | Przykład modelu |
|-----------|------|----------------|
| **Frame-level** | Klasyfikuj każdą klatkę osobno, agreguj | CNN + głosowanie |
| **3D Convolutions** | Konwolucja 3D (czas × przestrzeń) | C3D, I3D, SlowFast |
| **CNN + RNN/LSTM** | CNN = ekstraktor cech, LSTM = modelowanie czasowe | CNN-LSTM |
| **Video Transformers** | Attention w przestrzeni i czasie | TimeSformer, ViViT, VideoMAE |
| **Two-Stream** | Osobne sieci na RGB i optical flow | Two-Stream Networks |

### 6.3 Konwolucja 3D

W porównaniu z 2D convolution ($k \times k$), konwolucja 3D ma kernel $k_t \times k_h \times k_w$:

$$\text{out}(t, x, y) = \sum_{\tau=0}^{k_t-1} \sum_{i=0}^{k_h-1} \sum_{j=0}^{k_w-1} w(\tau, i, j) \cdot \text{in}(t+\tau, x+i, y+j)$$

**SlowFast Networks:**
- **Slow pathway:** niska częstotliwość klatek (np. co 16 klatka), duża sieć — rozpoznawanie scen
- **Fast pathway:** wysoka częstotliwość, mały model — wykrywanie ruchu

### 6.4 Video Transformers

**TimeSformer** (2021): Rozdziela attention czasowy i przestrzenny:
1. **Space Attention:** attention między patchami w jednej klatce
2. **Time Attention:** attention między tymi samymi patchami w różnych klatkach

**ViViT** (2021): 4 warianty factoryzacji spatio-temporal attention.

**VideoMAE** (2022): Self-supervised pretraining — maskuj 90% patcho-klatek, rekonstruuj.

### 6.5 Optical Flow

**Optical flow** — wektor przesunięcia pikseli między klatkami:

$$\mathbf{u}(x, y) = (u_x, u_y) \quad \text{(ruch piksela z klatki } t \text{ do } t+1\text{)}$$

Używany jako dodatkowy input (TVL1, RAFT) — informacja o ruchu.

### 6.6 Praktyczne przetwarzanie wideo

```python
import cv2
import torch

# Ekstrakcja klatek z pliku wideo
cap = cv2.VideoCapture("video.mp4")
frames = []
while cap.isOpened():
 ret, frame = cap.read()
 if not ret:
 break
 frame = cv2.resize(frame, (224, 224))
 frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
 frames.append(frame)
cap.release()

# Konwersja na tensor: (T, H, W, C) → (T, C, H, W) → (1, C, T, H, W)
video_tensor = torch.tensor(frames).permute(0, 3, 1, 2).float() / 255.0
video_tensor = video_tensor.unsqueeze(0).permute(0, 2, 1, 3, 4) # (B, C, T, H, W)
print(f"Video tensor shape: {video_tensor.shape}")
```

### 6.7 Zastosowania

| Zadanie | Input | Output |
|---------|-------|--------|
| **Action Recognition** | Klip wideo | Klasa akcji (bieganie, pływanie...) |
| **Video Classification** | Wideo | Kategoria treści |
| **Temporal Localization** | Wideo | Start/end akcji + klasa |
| **Video Captioning** | Wideo | Opis tekstowy |
| **Video QA** | Wideo + pytanie | Odpowiedź |

**Kluczowy wniosek:** Wideo = sekwencja obrazów + wymiar czasowy. Można zastosować:
- CNN (per frame) + agregacja (pooling / RNN / Transformer)
- Natywne modele 3D (C3D, SlowFast)
- Video Transformers (TimeSformer, ViViT)

---

## Zadania do samodzielnego rozwiązania

---

### Zadanie 1: DCGAN na Fashion-MNIST (trudne)

Zaimplementuj **DCGAN** (Deep Convolutional GAN):
1. Generator: ConvTranspose2d (z → 64 → 32 → 1)
2. Dyskryminator: Conv2d (1 → 32 → 64 → 1)
3. BatchNorm w obu (oprócz ostatniej warstwy D i pierwszej G)
4. LeakyReLU w D, ReLU w G, Tanh na wyjściu G
5. Trenuj 20 epok, pokaż wygenerowane obrazy co 5 epok

In [ ]:
# Zadanie 1: DCGAN
# TWÓJ KOD TUTAJ

### Zadanie 2: CLIP Zero-Shot (średnie)

Użyj pretrained CLIP do zero-shot classification:
1. Wczytaj 10 obrazów z CIFAR-10 (po jednym z każdej klasy)
2. Stwórz 10 promptów: `"a photo of a {class}"` 
3. Dla każdego obrazu oblicz similarity z każdym promptem
4. Oblicz accuracy zero-shot
5. Eksperymentuj z różnymi szablonami promptów (np. `"a blurry photo of a {class}"`)

In [ ]:
# Zadanie 2: CLIP zero-shot
# TWÓJ KOD TUTAJ

### Zadanie 3: Forward Diffusion Process (średnie)

1. Zaimplementuj cosine noise schedule (zamiast linear)
2. Porównaj wizualnie: linear vs cosine schedule na tym samym obrazie w krokach [0, 100, 300, 500, 700, 999]
3. Narysuj krzywe $\bar{\alpha}_t$ dla obu schedules

Cosine schedule:
$$\bar{\alpha}_t = \frac{f(t)}{f(0)}, \quad f(t) = \cos\left(\frac{t/T + s}{1+s} \cdot \frac{\pi}{2}\right)^2, \quad s=0.008$$

In [ ]:
# Zadanie 3: Cosine schedule
# TWÓJ KOD TUTAJ

### Zadanie 4: Contrastive Loss od zera (trudne)

Zaimplementuj prostą wersję contrastive learning:
1. Encoder: ResNet-18 (pretrained, bez FC layer)
2. Projection head: Linear(512, 128)
3. Stwórz pary augmentacji z SimCLR transforms
4. Zaimplementuj NT-Xent loss
5. Trenuj 5 epok na CIFAR-10 (bez etykiet!)
6. Ewaluacja: linear probing (zamroź encoder, trenuj linear classifier)

In [ ]:
# Zadanie 4: Contrastive learning
# TWÓJ KOD TUTAJ

### Zadanie 5: CLIP zero-shot classification (srednie)

Uzyj modelu CLIP do klasyfikacji obrazow **bez treningu**:

1. Wygeneruj 5 prostych obrazow (np. kolorowe prostokaty, kola)
2. Zdefiniuj 5 opisow tekstowych odpowiadajacych obrazom
3. Oblicz podobienstwo CLIP (image embedding vs text embedding)
4. Pokaz macierz podobienstwa jako heatmape
5. Czy CLIP poprawnie dopasowuje opisy do obrazow?

In [ ]:
# Zadanie 5: CLIP zero-shot
# from transformers import CLIPProcessor, CLIPModel
import torch
import numpy as np
import matplotlib.pyplot as plt
# TWOJ KOD TUTAJ

### Zadanie 6: Prosty GAN na MNIST
Zaimplementuj prosty GAN (generator + discriminator jako MLP).
Generator: z(100) -> 256 -> 512 -> 784, tanh output.
Discriminator: 784 -> 512 -> 256 -> 1, sigmoid output.
Wytrenuj 20 epok i co 5 epok zapisz wygenerowane cyfry (grid 4x4).

Narysuj krzywe loss generatora i dyskryminatora.

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# TWOJ KOD TUTAJ


### Zadanie 7: Augmentacja z transformacjami geometrycznymi
Zaimplementuj pipeline augmentacji dla małego zbioru (100 obrazów z CIFAR-10):
1. Zastosuj: random rotation, horizontal flip, color jitter, cutout
2. Wytrenuj prosty CNN: (a) bez augmentacji, (b) z augmentacją
3. Ile augmentacja poprawia accuracy przy małym zbiorze treningowym?

In [ ]:
import torch
import torchvision.transforms as T
from torchvision import datasets
import matplotlib.pyplot as plt

# TWOJ KOD TUTAJ


---

## Dodatek OAI: Ćwiczenia w stylu olimpiadowym

### Kontekst olimpiadowy
Na OAI i IOAI coraz częściej pojawiają się zadania wymagające użycia pretrained modeli w trybie zero-shot lub fine-tuning. Kluczowe:
- Umiejętność **szybkiego załadowania** pretrained modelu (CLIP, DETR, Stable Diffusion)
- **Prompt engineering** dla modeli vision-language
- Rozumienie **representation learning** i transfer learning

### Ćwiczenie OAI-10A: Multi-modal Retrieval

Zbuduj system text-to-image retrieval:
1. Zakoduj 1000 obrazów CIFAR-10 przez CLIP image encoder
2. Przyjmij zapytanie tekstowe (np. `"red sports car"`)
3. Znajdź 5 najbardziej podobnych obrazów (cosine similarity)
4. Wyświetl wyniki z similarity scores

### Ćwiczenie OAI-10B: Conditional GAN

Rozszerz GAN z tego modułu o **warunkowanie na klasie**:
1. Dodaj embedding klasy (nn.Embedding) do generatora i dyskryminatora
2. Konkatenuj embedding z wejściem G i z cechami D
3. Trenuj na MNIST, generuj cyfry na żądanie

### Ćwiczenie OAI-10C: Denoising Diffusion

Wytrenuj prosty model dyfuzyjny na MNIST:
1. Użyj SimpleUNet z tego modułu (lub zmodyfikuj)
2. Trenuj predykcję szumu (MSE loss)
3. Zaimplementuj sampling loop (DDPM)
4. Wygeneruj 16 próbek i wyświetl

In [ ]:
# Ćwiczenie OAI-16A/B/C: Twój kod
# TWÓJ KOD TUTAJ